# Training curve analysis — the 8-class sweep

Every number here comes from the **current** `results/` tree: the 16 runs trained against the
active hierarchy, whose classified leaves are `pyramidal`, `putative_cge`,
`putative_parvalbumin`, `putative_somatostatin`, `thalamocortical`, `astrocyte`, `microglia`,
`oligo`. Nothing from the earlier 24-class or 20-class suites is read, and the old
`20260811/results/` snapshot next to this notebook is deliberately **not** on the search path —
those runs used a different label tree, so putting them on a shared axis would compare classes
that are not the same classes.

## The sweep is a factorial grid, not a list of runs

Four factors vary, and every run is one cell of the grid. The notebook discovers them from each
run's own `results/<run>.json` `args` block rather than parsing run names, so a factor cannot be
mislabelled by a naming convention drifting away from what was actually trained.

| Factor | Values | What it asks |
|---|---|---|
| **aggregation** | mean pool · MPNN L2 · GraphTransformer | the ladder of learned mixing before the readout — none, fixed local neighbour averaging, adjacency-biased global attention |
| **window radius** | 10 µm · 20 µm | the baseline's *own* aggregation parameter: how much of a gain is the method versus simply seeing more context |
| **classifier head** | linear probe · shared 4×128 ResNet trunk | whether a margin survives a head with capacity of its own, or was the readout standing in for a missing MLP |
| **encoder** | trainable · frozen | with the aggregator frozen at init, only the head learns — so this separates "message passing helps" from "a random projection into 128 dims helps" |

Mean pooling has **no frozen counterpart** by construction: `MeanReadout` has zero parameters,
so freezing it is the same model. The grid is therefore 3 × 2 × 2 trainable runs plus 2 × 2
frozen ones (MPNN and GT, linear head) = 16.

Two things that are *not* factors and are held fixed across all 16, which is what makes the
comparison apples-to-apples: the windows, the split, and the LCPN head structure are identical,
and `class_balance=sample` is on everywhere.

## Sections

Each factor of the grid gets its own section, with its own controlled contrast — nothing here
asks the reader to re-run a cell with a different parameter to see a different comparison.

1. **Headline grid** — every run at its own best epoch, pivoted so aggregation reads down and
   the remaining factors across.
2. **Aggregation method** — mean pool vs MPNN vs GraphTransformer, the project's own question.
3. **Window radius** — 10 µm vs 20 µm.
4. **Classifier head** — linear probe vs shared 4×128 ResNet trunk.
5. **Encoder** — trainable vs frozen aggregator.
6. **Training curves** — convergence and overfitting.
7. **Per-class** window recall / precision / F1, plus the signed delta against mean pooling.
8. **Class support** — the denominator behind every per-class bar.
9. **Confusion matrices** — the off-diagonal structure a scalar cannot show.
10. **Input ablations at evaluation time** (`results/eval_ablations/`) — the trained checkpoints
    re-evaluated under nine perturbations of their inputs, with mean pooling as the invariance
    control.
11. **Geometry-only (`--no-embeddings`) runs**, if any are on disk.

Sections 2–5 share one shape: a table of **paired** deltas, a signed delta bar chart, and the two
absolute levels behind each delta. A pair is only ever formed between runs that agree on every
other factor, so each is a controlled contrast rather than a marginal average over an unbalanced
grid — which matters here, since mean pooling has no frozen counterpart and a marginal would let
that hole leak into the encoder contrast.

Kernel: **segclr_db (.venv)** — needs pandas + matplotlib
(`scripts/sbatch/install_matplotlib.sh`).

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path("..")
RESULTS_DIR = REPO_ROOT / "results"
ABLATION_DIR = RESULTS_DIR / "eval_ablations"
MANIFEST_PATH = REPO_ROOT / "data" / "manifest.json"

# scripts/train_gnn.py prefixes every run directory with this; the rest of the
# name is the agg_tag plus enabled switches.
RUN_PREFIX = "gnn_lcpn_scratch_"

# Which metric selects each run's "best" epoch -- matches train_gnn.py's own
# checkpoint-selection criterion (best val window balanced accuracy: more
# stable than cell-level, which majority-votes only a few hundred val cells),
# so "best epoch" here means the same weights as the saved checkpoint_best.pt.
BEST_EPOCH_METRIC = "window_balanced_accuracy"

# Window-level averages over ~2.4M windows and is what the loss is shaped by;
# cell-level majority-votes those up to 466 cells and is the headline.
SCALAR_METRICS = [
    "window_accuracy", "window_balanced_accuracy", "window_macro_precision", "window_macro_f1",
    "cell_accuracy", "cell_balanced_accuracy", "cell_macro_precision", "cell_macro_f1",
]
HEADLINE_METRICS = ["window_balanced_accuracy", "window_macro_f1",
                    "cell_balanced_accuracy", "cell_macro_f1"]

ARCH_LABELS = {"mean": "mean pool", "mpnn": "MPNN L2", "graph_transformer": "GT L4H4"}
ARCH_ORDER = ["mean", "mpnn", "graph_transformer"]

# The factors that define one cell of the grid. Order matters only for the
# paired-delta index; every entry must exist as a column on the registry.
FACTORS = ["arch", "window_um", "head", "encoder", "inputs"]

plt.rcParams["figure.dpi"] = 110

In [ ]:
# ---------------------------------------------------------------------------
# Run discovery.
#
# Factors are read from each run's own results/<run>.json "args" block, which
# is what train_gnn.py actually parsed, rather than re-derived from the run
# name. A name-based fallback covers a run that has started writing
# epoch_metrics.csv but has not yet written its end-of-run summary -- normal
# mid-sweep, and worth showing rather than silently dropping.
# ---------------------------------------------------------------------------

def _factors_from_args(args: dict) -> dict:
    return {
        "arch": args["architecture"],
        "window_um": int(round(float(args["window_nm"]) / 1000.0)),
        "head": "resnet" if args.get("cls_resnet") else "linear",
        "encoder": "frozen" if args.get("freeze_aggregator") else "trainable",
        "inputs": "geometry only" if args.get("no_embeddings") else "embeddings",
    }


def _factors_from_name(run: str) -> dict | None:
    """Fallback for a run still training. 10um is untagged, being the default."""
    tag = run[len(RUN_PREFIX):] if run.startswith(RUN_PREFIX) else run
    if tag.startswith("meanpool"):
        arch = "mean"
    elif tag.startswith("mpnn"):
        arch = "mpnn"
    elif tag.startswith("gt_"):
        arch = "graph_transformer"
    else:
        return None
    radius = re.search(r"_w(\d+)um", tag)
    return {
        "arch": arch,
        "window_um": int(radius.group(1)) if radius else 10,
        "head": "resnet" if "_resnet" in tag else "linear",
        "encoder": "frozen" if "frozenagg" in tag else "trainable",
        "inputs": "geometry only" if "_noemb" in tag else "embeddings",
    }


def run_label(row) -> str:
    bits = [ARCH_LABELS.get(row["arch"], row["arch"]), f"{row['window_um']}um"]
    if row["head"] != "linear":
        bits.append(row["head"])
    if row["encoder"] != "trainable":
        bits.append(row["encoder"])
    if row["inputs"] != "embeddings":
        bits.append("geom only")
    return " · ".join(bits)


def build_registry(results_dir=RESULTS_DIR) -> pd.DataFrame:
    rows = []
    for csv_path in sorted(results_dir.glob("*/epoch_metrics.csv")):
        run = csv_path.parent.name
        summary = results_dir / f"{run}.json"
        factors, source = None, "name"
        if summary.exists():
            args = json.loads(summary.read_text()).get("args")
            if args is not None:
                factors, source = _factors_from_args(args), "args"
        if factors is None:
            factors = _factors_from_name(run)
        if factors is None:
            print(f"  skipped {run}: no summary args and the name is not a known agg_tag")
            continue
        rows.append({
            "run": run, "csv": csv_path, "summary": summary,
            "has_summary": summary.exists(), "factors_from": source, **factors,
        })
    reg = pd.DataFrame(rows)
    if reg.empty:
        return reg
    reg["arch_rank"] = reg["arch"].map({a: i for i, a in enumerate(ARCH_ORDER)}).fillna(99)
    reg["label"] = reg.apply(run_label, axis=1)
    return reg.sort_values(
        ["inputs", "window_um", "head", "encoder", "arch_rank"],
        ascending=[False, True, True, True, True],
    ).reset_index(drop=True)


registry = build_registry()
WINDOWS_UM = sorted(registry["window_um"].unique()) if not registry.empty else []

unfinished = registry.loc[~registry["has_summary"], "run"].tolist()
if unfinished:
    print("still training (metrics CSV but no end-of-run summary; no confusion matrix): "
          + ", ".join(unfinished))
if registry.empty:
    raise RuntimeError(
        f"no runs under {RESULTS_DIR}. Every section below reads from the registry, so "
        "there is nothing to plot until a run writes results/<run>/epoch_metrics.csv."
    )
print(f"{len(registry)} run(s) discovered under {RESULTS_DIR}")
display(registry[["run", "label", "arch", "window_um", "head", "encoder", "inputs",
                  "factors_from"]])

In [ ]:
# ---------------------------------------------------------------------------
# Shared helpers.
# ---------------------------------------------------------------------------

def load_metrics(csv_path) -> pd.DataFrame:
    return pd.read_csv(csv_path)


def class_names_from_columns(df: pd.DataFrame, prefix: str = "window_recall_") -> list[str]:
    return [c[len(prefix):] for c in df.columns if c.startswith(prefix)]


def best_epoch_row(df: pd.DataFrame, metric: str = BEST_EPOCH_METRIC) -> pd.Series:
    return df.loc[df[metric].idxmax()]


def f1_from_pr(precision, recall) -> np.ndarray:
    """Per-class F1 from precision + recall -- not logged to the CSV (only the
    macro F1 scalar is), so every plot that wants per-class F1 derives it here
    instead. Matches gnn/metrics.py::macro_f1's harmonic-mean formula, just
    element-wise over classes instead of pre-averaged."""
    p, r = np.asarray(precision, dtype=float), np.asarray(recall, dtype=float)
    denom = p + r
    return np.divide(2 * p * r, denom, out=np.zeros_like(denom), where=denom > 0)


def best_table(reg: pd.DataFrame, metric: str = BEST_EPOCH_METRIC) -> pd.DataFrame:
    """One row per run, taken at that run's own best epoch."""
    rows = []
    for _, r in reg.iterrows():
        d = load_metrics(r["csv"])
        b = best_epoch_row(d, metric)
        rows.append({**r.to_dict(), "best_epoch": int(b["epoch"]),
                     "n_epochs": int(d["epoch"].max()) + 1,
                     **{c: float(b[c]) for c in SCALAR_METRICS if c in b.index}})
    return pd.DataFrame(rows)


def select(reg: pd.DataFrame, **filters) -> pd.DataFrame:
    sel = reg
    for key, value in filters.items():
        sel = sel[sel[key].isin(value)] if isinstance(value, (list, tuple, set)) else \
            sel[sel[key] == value]
    return sel


def group_pairs(reg: pd.DataFrame, **filters) -> list[tuple[str, Path]]:
    """(label, csv path) for the runs matching `filters`, in aggregation order."""
    sel = select(reg, **filters).sort_values(["arch_rank", "encoder", "head"])
    return [(row["label"], row["csv"]) for _, row in sel.iterrows()]


def grouped_bar(ax, x_labels, series_by_label, colors=None, ylabel="score", title="",
                ylim=(0, 1)):
    """series_by_label: {series_label: [value per x_label]}. One group of
    adjacent, differently-colored bars per x position."""
    n_series = len(series_by_label)
    x = np.arange(len(x_labels))
    width = 0.8 / max(n_series, 1)
    if colors is None:
        colors = plt.cm.tab10(np.linspace(0, 1, max(n_series, 2)))
    for i, (label, values) in enumerate(series_by_label.items()):
        offset = i * width - (n_series - 1) * width / 2
        ax.bar(x + offset, values, width, label=label, color=colors[i])
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=60 if len(x_labels) > 4 else 0,
                       ha="right" if len(x_labels) > 4 else "center")
    ax.set_ylabel(ylabel)
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_title(title)
    ax.legend(fontsize=8)


# ---------------------------------------------------------------------------
# The groups the per-class, curve and confusion sections iterate over. Built
# from the registry rather than hand-listed, so a run added to the sweep shows
# up everywhere at once and a run that was never trained cannot leave a stale
# entry behind.
#
# Aggregation groups hold one run per architecture with every other factor
# pinned, which is what makes mean pooling a legitimate reference point inside
# them -- hence DELTA_GROUPS.
# ---------------------------------------------------------------------------
AGG_GROUPS, ENC_GROUPS, GEOM_GROUPS, DELTA_GROUPS = [], [], [], []

for _um in WINDOWS_UM:
    for _head in ("linear", "resnet"):
        _pairs = group_pairs(registry, window_um=_um, head=_head,
                             encoder="trainable", inputs="embeddings")
        if len(_pairs) > 1:
            _name = f"Aggregation @ {_um}um, {_head} head"
            AGG_GROUPS.append((_name, _pairs))
            _mean = [lab for lab, _ in _pairs if lab.startswith(ARCH_LABELS["mean"])]
            if _mean:
                DELTA_GROUPS.append((_name, _mean[0]))

for _um in WINDOWS_UM:
    _pairs = group_pairs(registry, window_um=_um, head="linear", inputs="embeddings")
    if any(lab.endswith("frozen") for lab, _ in _pairs):
        ENC_GROUPS.append((f"Encoder trainable vs frozen @ {_um}um, linear head", _pairs))

_geom = group_pairs(registry, inputs="geometry only")
if _geom:
    GEOM_GROUPS.append(("Geometry only (--no-embeddings)", _geom))

# The union, for the sections that walk every group once (curves, per-class,
# confusion matrices) rather than isolating one factor.
COMPARISON_GROUPS = AGG_GROUPS + ENC_GROUPS + GEOM_GROUPS

for _name, _pairs in COMPARISON_GROUPS:
    print(f"{_name}: {', '.join(lab for lab, _ in _pairs)}")


# ---------------------------------------------------------------------------
# One factor's controlled contrast. Sections 2-5 each call this once with the
# factor fixed in the call, so the comparison a section makes is visible in the
# notebook itself rather than being selected by a variable the reader edits.
# ---------------------------------------------------------------------------

def paired_deltas(best_df, factor, before, after, metrics=HEADLINE_METRICS):
    """Signed change in `metrics` for every pair of runs that differ in
    `factor` alone. Returns one row per surviving configuration, carrying both
    absolute levels alongside the difference."""
    others = [f for f in FACTORS if f != factor]
    left = best_df[best_df[factor] == before].drop_duplicates(others).set_index(others)
    right = best_df[best_df[factor] == after].drop_duplicates(others).set_index(others)
    keys = left.index.intersection(right.index)
    rows = []
    for key in keys:
        a, b = left.loc[key], right.loc[key]
        key_tuple = key if isinstance(key, tuple) else (key,)
        row = dict(zip(others, key_tuple))
        bits = []
        if "arch" in row:
            bits.append(ARCH_LABELS.get(row["arch"], row["arch"]))
        if "window_um" in row:
            bits.append(f"{row['window_um']}um")
        for extra in ("head", "encoder"):
            if extra in row:
                bits.append(str(row[extra]))
        row["config"] = " · ".join(bits) or "all"
        row["from"], row["to"] = a["run"], b["run"]
        for m in metrics:
            row[f"{m}_before"], row[f"{m}_after"] = float(a[m]), float(b[m])
            row[f"d_{m}"] = float(b[m]) - float(a[m])
        rows.append(row)
    out = pd.DataFrame(rows)
    return out.sort_values("config").reset_index(drop=True) if not out.empty else out


def factor_contrast(best_df, title, factor, before, after,
                    before_label=None, after_label=None, metrics=HEADLINE_METRICS):
    """Table + signed delta bars + the absolute levels behind them, for one
    factor. Returns the delta frame, or None when the contrast is not on disk."""
    before_label = before_label or str(before)
    after_label = after_label or str(after)
    have = set(best_df[factor])
    if before not in have or after not in have:
        print(f"[{title}] one side of the contrast is not on disk -- skipped.")
        return None
    d = paired_deltas(best_df, factor, before, after, metrics)
    if d.empty:
        print(f"[{title}] no run pairs match on every other factor -- skipped.")
        return None

    print(f"{title}: {len(d)} matched pair(s), {before_label} -> {after_label}")
    display(d[["config", "from", "to"] + [f"d_{m}" for m in metrics]].round(4)
            .style.hide(axis="index"))

    short = ["window bal. acc", "window macro F1", "cell bal. acc", "cell macro F1"][:len(metrics)]
    series = {row["config"]: [row[f"d_{m}"] for m in metrics] for _, row in d.iterrows()}
    lim = max(0.01, float(np.abs(np.concatenate(
        [np.asarray(v) for v in series.values()])).max()) * 1.2)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    grouped_bar(ax, short, series, ylabel=f"delta ({after_label} - {before_label})",
                title=f"{title}: {before_label} -> {after_label}", ylim=(-lim, lim))
    ax.axhline(0, color="black", linewidth=0.8)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # The levels behind the deltas: a +0.01 off a weak model and off a strong
    # one are not the same result.
    configs = d["config"].tolist()
    for m, nice in zip(metrics[:2], short[:2]):
        levels = {before_label: d[f"{m}_before"].tolist(),
                  after_label: d[f"{m}_after"].tolist()}
        lo = min(min(v) for v in levels.values())
        fig, ax = plt.subplots(figsize=(max(9, len(configs) * 1.6), 4.2))
        grouped_bar(ax, configs, levels, ylabel=nice,
                    title=f"{title}: {nice}, absolute", ylim=(max(0.0, lo - 0.05), 1.0))
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()
    return d

## 1. Headline grid

Each run at its **own** best epoch. Window-level metrics average over ~2.4M test windows;
cell-level metrics majority-vote those up to 466 cells and are correspondingly noisier — which
is exactly why checkpoint selection uses the window metric.

Window *count* is the same at both radii (one window per node); only window *size* changes
(mean 10.7 nodes at 10 µm, 22.1 at 20 µm). So a radius effect is not a sample-size effect.

Under this class imbalance raw accuracy is not a sufficient result — balanced accuracy and macro
F1 are the columns to read.

In [ ]:
best = best_table(registry)
_cols = ["label", "run", "best_epoch", "n_epochs"] + SCALAR_METRICS
display(best[_cols].round(4).style.hide(axis="index"))

In [ ]:
# The pivot the grid is actually for: aggregation down, the remaining factors
# across. Frozen-encoder runs sit in their own columns rather than being
# averaged into the trainable ones.
def grid_pivot(best_df, metric):
    df = best_df.copy()
    df["config"] = (df["window_um"].astype(str) + "um / " + df["head"]
                    + df["encoder"].map({"trainable": "", "frozen": " / frozen"})
                    + df["inputs"].map({"embeddings": "", "geometry only": " / geom"}))
    df["aggregation"] = df["arch"].map(ARCH_LABELS)
    piv = df.pivot_table(index="aggregation", columns="config", values=metric)
    return piv.reindex([ARCH_LABELS[a] for a in ARCH_ORDER if ARCH_LABELS[a] in piv.index])


for _metric in HEADLINE_METRICS:
    print(f"### {_metric}")
    display(grid_pivot(best, _metric).round(4))

## 2. Aggregation method

The project's own question: does a GNN over the window's local subgraph beat a geodesic mean over
the same window? Every other factor is pinned, so each group below differs by the aggregation
method alone — same windows, same split, same LCPN head, same radius, same classifier head.

One caveat that belongs here rather than in a footnote: the three architectures do **not** see the
same node features. `mean` and `mpnn` consume the raw 64-dim embeddings only, while the
GraphTransformer additionally receives the Laplacian PE and the 4-channel center-relative offset
(`SAGEConv` discards `edge_attr`, so the MPNN sees geometry only as topology). A GT-over-MPNN
margin is therefore not attributable to attention alone.

In [ ]:
for group_name, pairs in AGG_GROUPS:
    labels = [lab for lab, _ in pairs]
    sub = best[best["label"].isin(labels)].set_index("label").reindex(labels)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    series = {lab: [sub.loc[lab, m] for m in HEADLINE_METRICS] for lab in labels}
    grouped_bar(ax, ["window bal. acc", "window macro F1", "cell bal. acc", "cell macro F1"],
                series, title=f"{group_name} (each at its own best epoch)")
    ax.set_ylim(0.5, 1.0)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Against mean pooling, paired on radius / head / encoder / inputs.
agg_deltas = {
    "MPNN vs mean pooling": factor_contrast(
        best, "Aggregation", "arch", "mean", "mpnn",
        before_label=ARCH_LABELS["mean"], after_label=ARCH_LABELS["mpnn"]),
    "GraphTransformer vs mean pooling": factor_contrast(
        best, "Aggregation", "arch", "mean", "graph_transformer",
        before_label=ARCH_LABELS["mean"], after_label=ARCH_LABELS["graph_transformer"]),
    "GraphTransformer vs MPNN": factor_contrast(
        best, "Aggregation", "arch", "mpnn", "graph_transformer",
        before_label=ARCH_LABELS["mpnn"], after_label=ARCH_LABELS["graph_transformer"]),
}

## 3. Window radius: 10 µm vs 20 µm

The radius is the *baseline's own* aggregation parameter, which makes this the sharpest control on
the section-2 claim: a GNN margin that is smaller than simply widening the mean-pool window is a
weaker result than it looks.

Window *count* is identical at both radii — one window per node — so this is not a sample-size
effect; only window *size* changes (mean 10.7 nodes at 10 µm, 22.1 at 20 µm).

In [ ]:
radius_delta = factor_contrast(best, "Window radius", "window_um", 10, 20,
                              before_label="10um", after_label="20um")

## 4. Classifier head: linear probe vs ResNet 4×128

By default the LCPN's per-node heads sit directly on the readout embedding — a linear probe.
`--cls-resnet` inserts `gnn/resnet.py::DeepResNetTrunk` (4 layers × 128, the lab's own
`local_classifier_sngp.yaml` defaults) between readout and heads, shared across all nodes.

This factor is orthogonal to aggregation, which is what makes the contrast worth its own section:
if a margin between aggregators shrinks once the head has capacity of its own, the readout was
partly standing in for a missing MLP rather than encoding structure the mean could not reach.

In [ ]:
head_delta = factor_contrast(best, "Classifier head", "head", "linear", "resnet",
                            before_label="linear probe", after_label="ResNet 4x128")

## 5. Encoder: trainable vs frozen

With the aggregator frozen at initialization, only the classifier head learns — so this separates
"message passing helps" from "a random projection into the readout dimension helps". A frozen GNN
that still beats mean pooling is measuring the projection, not the learned mixing.

Mean pooling has **no frozen counterpart**: `MeanReadout` has zero parameters, so freezing it is
the same model. Pairing is what keeps that hole out of the contrast.

In [ ]:
for group_name, pairs in ENC_GROUPS:
    labels = [lab for lab, _ in pairs]
    sub = best[best["label"].isin(labels)].set_index("label").reindex(labels)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    series = {lab: [sub.loc[lab, m] for m in HEADLINE_METRICS] for lab in labels}
    grouped_bar(ax, ["window bal. acc", "window macro F1", "cell bal. acc", "cell macro F1"],
                series, title=f"{group_name} (each at its own best epoch)")
    ax.set_ylim(0.5, 1.0)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

encoder_delta = factor_contrast(best, "Encoder", "encoder", "trainable", "frozen")

## 6. Training curves

Train loss and validation window balanced accuracy against epoch, one line per run, grouped the
same way as section 1. This is where convergence and overfitting show up: a run whose train loss
keeps falling while its validation metric flattens has stopped generalizing, and a run still
climbing at the last epoch was cut short rather than converged.

Note "val" is an alias for the test split (see `data/build_dataset_from_store.py`), so these
curves are not held out from the reported numbers — an accepted trade-off of the two-way split.

In [ ]:
for group_name, pairs in COMPARISON_GROUPS:
    fig, (ax_loss, ax_bacc) = plt.subplots(1, 2, figsize=(13, 4.5))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(pairs), 2)))
    for (label, path), color in zip(pairs, colors):
        d = load_metrics(path)
        ax_loss.plot(d["epoch"], d["train_loss"], label=label, color=color)
        ax_bacc.plot(d["epoch"], d["window_balanced_accuracy"], label=label, color=color)
    ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("train loss")
    ax_loss.set_title(f"{group_name}: train loss")
    ax_bacc.set_xlabel("epoch"); ax_bacc.set_ylabel("val window balanced accuracy")
    ax_bacc.set_title(f"{group_name}: val window balanced accuracy")
    for a in (ax_loss, ax_bacc):
        a.legend(fontsize=8)
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. Per-class comparison

Per-class window-level recall, precision and F1 at each run's best epoch. With support spanning
several orders of magnitude (section 8), this is where an apparently healthy headline number
turns out to rest on the populous classes alone.

For each aggregation group a second view follows: the signed **delta against mean pooling** in
that same cell of the grid. Absolute bars are dominated by the classes every model gets right
and bury the effect being measured; the difference from the baseline aggregator is the effect.

In [ ]:
def per_class_series(pairs):
    """{label: [value per class]} for recall / precision / F1, plus the class list."""
    classes = class_names_from_columns(load_metrics(pairs[0][1]))
    recall, precision, f1 = {}, {}, {}
    for label, path in pairs:
        d = load_metrics(path)
        b = best_epoch_row(d)
        recall[label] = [float(b[f"window_recall_{c}"]) for c in classes]
        if "window_macro_precision" in d.columns:
            precision[label] = [float(b[f"window_precision_{c}"]) for c in classes]
            f1[label] = f1_from_pr(precision[label], recall[label])
    return classes, recall, precision, f1


for group_name, pairs in COMPARISON_GROUPS:
    classes, recall, precision, f1 = per_class_series(pairs)
    w = max(10, len(classes) * 0.9)
    for series, ylabel in ((recall, "recall"), (precision, "precision"), (f1, "F1")):
        if len(series) != len(recall):
            print(f"note: [{group_name}] a run predates per-class {ylabel} logging -- skipped.")
            continue
        fig, ax = plt.subplots(figsize=(w, 4.5))
        grouped_bar(ax, classes, series, ylabel=f"window-level {ylabel}",
                    title=f"{group_name}: per-class window {ylabel}")
        plt.tight_layout()
        plt.show()

In [ ]:
# Delta view -- aggregation groups only, against mean pooling in the same cell
# of the grid (DELTA_GROUPS). Every entry there differs from the reference by
# the aggregation method alone.
_pairs_by_name = dict(COMPARISON_GROUPS)

for group_name, baseline_label in DELTA_GROUPS:
    pairs = _pairs_by_name[group_name]
    classes, recall, _precision, f1 = per_class_series(pairs)
    for series, ylabel in ((recall, "recall"), (f1, "F1")):
        if baseline_label not in series or len(series) < 2:
            continue
        base = np.asarray(series[baseline_label], dtype=float)
        deltas = {lab: np.asarray(v, dtype=float) - base
                  for lab, v in series.items() if lab != baseline_label}
        lim = max(0.01, float(np.abs(np.concatenate(list(deltas.values()))).max()) * 1.15)
        fig, ax = plt.subplots(figsize=(max(10, len(classes) * 0.9), 4.5))
        grouped_bar(ax, classes, deltas, ylabel=f"delta window {ylabel}",
                    title=f"{group_name}: per-class window {ylabel} vs {baseline_label}",
                    ylim=(-lim, lim))
        ax.axhline(0, color="black", linewidth=0.8)
        plt.tight_layout()
        plt.show()

## 8. Class support (train-split window counts)

Context for every per-class bar above — a class with few training windows can swing wildly on
recall from run to run purely from noise, which the recall number alone doesn't show.

Counts are folded up through the **active** hierarchy via `data/dataset_lcpn.py::load_hierarchy`,
not re-derived here: `manifest.json` stores each cell's granular `cell_type`, while these runs
classify at a coarser level, so keying the granular counts against the coarse class names would
match nothing and report zero support everywhere. A class printed as missing has no train
support at all and cannot be learned — for this dataset that is an embedding-availability gap,
not a labelling one.

In [ ]:
import sys

sys.path.insert(0, str(REPO_ROOT.resolve()))
from data.dataset_lcpn import load_hierarchy  # noqa: E402


def train_window_counts_by_class(manifest_path) -> dict[str, int]:
    manifest = json.loads(Path(manifest_path).read_text())
    hierarchy = load_hierarchy(manifest)
    counts: dict[str, int] = {}
    for info in manifest["cells"].values():
        if info["split"] != "train":
            continue
        path = hierarchy.label_paths.get(info["cell_type"])
        if path is None:
            continue
        counts[path[-1]] = counts.get(path[-1], 0) + info["n_nodes_covered"]
    return counts


classes = class_names_from_columns(load_metrics(registry.iloc[0]["csv"]))
support = train_window_counts_by_class(MANIFEST_PATH)
missing = [c for c in classes if not support.get(c)]
if missing:
    print(f"note: no train support for {missing} -- these cannot be learned.")

fig, ax = plt.subplots(figsize=(max(8, len(classes) * 0.9), 4))
ax.bar(classes, [support.get(c, 0) for c in classes], color="#55A868")
ax.set_yscale("log")
ax.set_ylabel("train window count (log scale)")
ax.set_xticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=60, ha="right")
ax.set_title("Class support (train split)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Confusion matrices

One heatmap per run, gridded per comparison group, so a whole sweep reads at a glance.

These come from `results/<run>.json`, not `epoch_metrics.csv`. The CSV logs only scalars and
per-class recall/precision, and a confusion matrix cannot be reconstructed from those — recall
fixes the row sums and precision the column sums, but neither pins down the off-diagonal mass.
The JSON matrices are the **best-epoch checkpoint's** predictions on the test split:
`scripts/train_gnn.py` reloads `checkpoint_best.pt` before its final evaluation, so these agree
with the best-epoch rows in section 1 rather than with the last epoch.

Rows are normalized to sum to 1, which makes the diagonal exactly per-class recall and lets every
panel share one 0–1 colour scale. That sharing is the point of a grid: raw counts are not
comparable across panels, because the largest class holds most of the mass and would saturate its
own row in every model. The number under each title is balanced accuracy recomputed from the
matrix (mean diagonal over classes that have support), as a check that the panel and section 1
agree.

Off-diagonal structure is what a scalar cannot show. Two models with identical balanced accuracy
can fail in completely different ways — one spreading errors evenly, another collapsing a whole
family into its most populous sibling — and with an LCPN that difference is usually the
interesting part, since a top-down cascade can only recover from a mistake made above it by
accident.

In [ ]:
# Cell-level ("test_metrics") is the headline; window-level is the diagnostic.
CM_GRANULARITIES = [("cell", "test_metrics"), ("window", "window_test_metrics")]
CM_ANNOTATE_MAX_CLASSES = 12
CM_COLORMAP = "magma"


def summary_path_for(csv_path):
    """results/<run>/epoch_metrics.csv -> results/<run>.json. Built from the
    directory name rather than Path.with_suffix, which would truncate at the
    first dot in a run name."""
    run_dir = Path(csv_path).parent
    return run_dir.parent / f"{run_dir.name}.json"


def load_confusion(csv_path, key):
    """(matrix, classes), or (None, None) if the run has not finished its final
    evaluation yet -- a normal mid-sweep state, not an error."""
    path = summary_path_for(csv_path)
    if not path.exists():
        return None, None
    payload = json.loads(path.read_text())
    cm = (payload.get(key) or {}).get("confusion_matrix")
    if cm is None:
        return None, None
    return np.asarray(cm, dtype=float), payload.get("classes")


def row_normalize(cm):
    """Rows -> per-class recall. Classes with no test support stay all-zero
    rather than dividing by zero, and are excluded from the mean diagonal."""
    totals = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, totals, out=np.zeros_like(cm), where=totals > 0)


def confusion_grid(pairs, group_name, key, granularity):
    loaded, pending = [], []
    for label, path in pairs:
        cm, classes = load_confusion(path, key)
        if cm is None:
            pending.append(label)
        else:
            loaded.append((label, cm, classes))
    if pending:
        print(f"  [{group_name}] no {granularity}-level matrix yet: {', '.join(pending)}")
    if not loaded:
        return

    n_classes = len(loaded[0][2])
    ncols = min(3, len(loaded))
    nrows = -(-len(loaded) // ncols)  # ceil
    panel = max(3.0, 0.42 * n_classes)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel * ncols + 1.4, panel * nrows),
                             squeeze=False)
    flat = axes.ravel()

    image = None
    for ax, (label, cm, classes) in zip(flat, loaded):
        norm = row_normalize(cm)
        supported = cm.sum(axis=1) > 0
        bacc = float(np.diag(norm)[supported].mean()) if supported.any() else float("nan")
        image = ax.imshow(norm, cmap=CM_COLORMAP, vmin=0.0, vmax=1.0)
        ax.set_title(f"{label}\nbalanced acc {bacc:.3f}", fontsize=9)
        ax.set_xticks(range(len(classes)))
        ax.set_yticks(range(len(classes)))
        ax.set_xticklabels(classes, rotation=90, fontsize=7)
        ax.set_yticklabels(classes, fontsize=7)
        ax.set_xlabel("predicted", fontsize=8)
        ax.set_ylabel("true", fontsize=8)
        if len(classes) <= CM_ANNOTATE_MAX_CLASSES:
            for i in range(len(classes)):
                for j in range(len(classes)):
                    v = norm[i, j]
                    if v > 0.005:
                        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                                color="white" if v < 0.6 else "black")

    for ax in flat[len(loaded):]:
        ax.axis("off")

    fig.suptitle(f"{group_name}: {granularity}-level confusion (row-normalized)", fontsize=11)
    fig.colorbar(image, ax=axes, fraction=0.02, pad=0.02, label="fraction of true class")
    plt.show()


for granularity, key in CM_GRANULARITIES:
    for group_name, pairs in COMPARISON_GROUPS:
        confusion_grid(pairs, group_name, key, granularity)

## 10. Input ablations at evaluation time

Sections 1–9 compare models that were *trained* differently. This section is different in kind:
it takes the already-trained checkpoints and re-evaluates each under nine perturbations of its
inputs (`scripts/eval_ablations.py` → `results/eval_ablations/ablations*.csv`), asking whether a
model that scores well actually *uses* the arrangement of nodes or only their multiset.

| condition | what it breaks |
|---|---|
| `identity` | nothing — the reference each delta is measured against |
| `recompute_lpe` | nothing structural; the Laplacian PE is recomputed on the *unmodified* graph, so this bounds how much of any rewire delta is just eigenvector sign/degeneracy convention. **Read it before reading `rewire`.** |
| `permute_x` | the correspondence between an embedding and its position, within the window. Graph, PE and geometry are preserved bit-for-bit |
| `rewire` | the topology, preserving the undirected edge count per window, with the PE recomputed to match |
| `rewire_lpe_stale` | same rewire, original PE retained — its gap from `rewire` is how much structure the model recovers from the PE alone |
| `rewire_lpe0` | the wrong-graph / no-PE corner |
| `drop_edges` | every edge — message passing has nothing to pass along |
| `zero_pos_enc` | the Laplacian PE |
| `zero_rel_pos` | the center-relative offset (all 4 channels) |

Every intervention is bounded by `batch.ptr`, so nothing leaks between windows belonging to
different cells.

**Mean pooling is the control, and it is a real one.** `MeanReadout` over raw embeddings is
permutation-invariant and reads no edges, so its predictions must be *exactly* unchanged by every
condition. A moving mean-pool row would mean an intervention escaped its window or the wrong
tensors reached a model — the harness would be broken, not the finding. The first cell below
checks this numerically rather than asserting it in prose.

**Coverage caveat: the GraphTransformer is not in this sweep.** `ablations.csv` and
`ablations_w20um.csv` hold the mean and MPNN checkpoints only, so nothing here speaks to
attention. The cells below read whatever runs are present rather than assuming a fixed list, so
re-running `scripts/sbatch/eval_ablations.sh` with the GT checkpoints in `RUNS` makes them appear
without editing the notebook.

In [ ]:
ABLATION_CSVS = sorted(p for p in ABLATION_DIR.glob("ablations*.csv")
                       if "limit" not in p.name)  # limit*batches files are pilot runs

if not ABLATION_CSVS:
    ablations = pd.DataFrame()
    print(f"no ablation sweeps under {ABLATION_DIR} -- section 7 is empty.")
else:
    ablations = pd.concat([pd.read_csv(p) for p in ABLATION_CSVS], ignore_index=True)
    ablations["window_um"] = (ablations["window_nm"] / 1000).round().astype(int)
    ablations["arch_label"] = ablations["architecture"].map(ARCH_LABELS)
    ablations["config"] = (
        ablations["arch_label"] + " · " + ablations["window_um"].astype(str) + "um"
        + np.where(ablations["cls_resnet"], " · resnet", "")
        + np.where(ablations["frozen_agg"], " · frozen", "")
    )
    print(f"loaded {', '.join(p.name for p in ABLATION_CSVS)}: "
          f"{ablations['run'].nunique()} run(s) x {ablations['condition'].nunique()} conditions")

    # Harness self-check: the permutation-invariant control must not move.
    control = ablations[ablations["architecture"] == "mean"]
    if control.empty:
        print("WARNING: no mean-pool rows -- the sweep has no invariance control.")
    else:
        worst = control["d_window_macro_f1"].abs().max()
        verdict = "OK" if worst < 1e-4 else "FAILED -- an intervention escaped its window"
        print(f"control check: max |delta window macro F1| over mean-pool rows = {worst:.2e}  [{verdict}]")

    missing_arch = sorted(set(ARCH_LABELS) - set(ablations["architecture"]))
    if missing_arch:
        print("not in this sweep: " + ", ".join(ARCH_LABELS[a] for a in missing_arch))

In [ ]:
# Delta-from-identity heatmap: conditions down, checkpoints across. The mean
# columns are the control and should read as a solid zero stripe.
CONDITION_ORDER = ["recompute_lpe", "permute_x", "rewire", "rewire_lpe_stale", "rewire_lpe0",
                   "drop_edges", "zero_pos_enc", "zero_rel_pos"]

if not ablations.empty:
    for level in ("window", "cell"):
        col = f"d_{level}_macro_f1"
        piv = ablations.pivot_table(index="condition", columns="config", values=col)
        piv = piv.reindex([c for c in CONDITION_ORDER if c in piv.index])
        piv = piv[sorted(piv.columns, key=lambda c: ("mean" not in c, c))]

        lim = float(np.nanmax(np.abs(piv.to_numpy()))) or 0.01
        fig, ax = plt.subplots(figsize=(1.5 * len(piv.columns) + 3.5, 0.55 * len(piv) + 2.2))
        im = ax.imshow(piv.to_numpy(), cmap="RdBu", vmin=-lim, vmax=lim, aspect="auto")
        ax.set_xticks(range(len(piv.columns)))
        ax.set_xticklabels(piv.columns, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(piv.index)))
        ax.set_yticklabels(piv.index, fontsize=8)
        for i in range(len(piv.index)):
            for j in range(len(piv.columns)):
                v = piv.to_numpy()[i, j]
                if np.isfinite(v):
                    ax.text(j, i, f"{v:+.3f}", ha="center", va="center", fontsize=7,
                            color="black" if abs(v) < 0.6 * lim else "white")
        ax.set_title(f"Change in {level}-level macro F1 vs the unperturbed model\n"
                     "(blue = worse under the perturbation; mean-pool columns are the control)",
                     fontsize=10)
        fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label=f"delta {level} macro F1")
        plt.tight_layout()
        plt.show()

In [ ]:
# Absolute view, so a large delta off a weak model is not mistaken for a large
# delta off a strong one.
if not ablations.empty:
    conditions = ["identity"] + [c for c in CONDITION_ORDER
                                 if c in set(ablations["condition"])]
    configs = sorted(ablations["config"].unique(), key=lambda c: ("mean" not in c, c))
    piv = ablations.pivot_table(index="condition", columns="config",
                                values="window_macro_f1").reindex(conditions)[configs]
    fig, ax = plt.subplots(figsize=(max(11, len(conditions) * 1.5), 5))
    grouped_bar(ax, conditions, {c: piv[c].tolist() for c in configs},
                ylabel="window macro F1",
                title="Window macro F1 under each input perturbation", ylim=(0, 1))
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 11. Geometry-only runs (`--no-embeddings`)

The complement of section 10's `permute_x`: instead of scrambling which node an embedding sits on,
`--no-embeddings` removes the SegCLR embedding from the node input entirely, leaving the graph,
the center-relative offset and the Laplacian PE. Paired against the embedding-carrying run at the
same radius, the gap is how much of a score is the embeddings and how much is the shape they sit
on.

**No such runs are in the current `results/` tree.** Earlier ones existed at 40 µm
(`logs/train_{gt,mpnn}_w40_noemb_*.out`) but their results were cleared along with the rest of the
pre-8-class suite, and 40 µm is not part of this sweep in any case — so pairing them against
anything here would compare across both a label tree and a radius. The cell below discovers
`inputs == "geometry only"` runs from the registry and pairs each with its twin automatically, so
it fills in the moment such a run lands; until then it prints exactly that.

In [ ]:
geom = select(registry, inputs="geometry only")
if geom.empty:
    print("no --no-embeddings runs on disk. Train one against this hierarchy to populate "
          "this section, e.g.\n"
          "  ARCHITECTURE=graph_transformer WINDOW_NM=10000 "
          "EXTRA_ARGS='--no-embeddings' \\\n"
          "    sbatch scripts/sbatch/train_gnn.sh\n"
          "then re-run this notebook.")
else:
    rows = []
    for _, g in geom.iterrows():
        twin = select(registry, inputs="embeddings", arch=g["arch"], window_um=g["window_um"],
                      head=g["head"], encoder=g["encoder"])
        if twin.empty:
            print(f"{g['run']}: no embedding-carrying twin at the same radius/head/encoder "
                  "-- shown without a pair.")
        names = twin["run"].tolist() + [g["run"]]
        pair = group_pairs(registry, run=names)
        rows.append(best_table(select(registry, run=names)))
        classes, recall, _precision, _f1 = per_class_series(pair)
        fig, ax = plt.subplots(figsize=(max(10, len(classes) * 0.9), 4.5))
        grouped_bar(ax, classes, recall, ylabel="window-level recall",
                    title=f"Geometry only vs embeddings: {ARCH_LABELS[g['arch']]} "
                          f"{g['window_um']}um")
        plt.tight_layout()
        plt.show()
    display(pd.concat(rows)[["label", "run"] + HEADLINE_METRICS].round(4)
            .style.hide(axis="index"))